# SLURM scripts (Altamira)

In [1]:
import pandas as pd

In [3]:
df_files = pd.read_csv("project_selection.csv", index_col=0)
df_files.head()

,SubSet,FileName,sample,model_selection
1,J120,j1201_4.sm,True,True
7,J120,j1204_9.sm,True,True
23,J120,j12014_10.sm,True,True
30,J120,j12020_3.sm,True,True
37,J120,j12022_10.sm,True,True


In [ ]:
# Create SLURM scripts for running a set of jobs at Altamira cloud computing
# Random Forest, XGBoost, Support Vector Machine, Elastic Net, EBM

for model in ['rf','xgb','sv','en','ebm','fastai']:
    # time limit for each model
    if model in ['rf', 'xgb','sv','en']:
        time="5:00:00"
    elif model in ['fastai', 'ebm']:
        time="10:00:00"
    for file_name in df_files['FileName'].values:
        project_name = file_name.split('.')[0]
        with open(f"jobs/slurm_lhs_model_selection_{model}_project{project_name}.sh", "w") as f:
            f.write("#!/bin/bash\n")
            f.write(f"#SBATCH --job-name=lhs_msel_{model}_{project_name}"+"\n")
            f.write(f"#SBATCH --output=lhs_msel_{model}_{project_name}.out"+"\n")
            f.write(f"#SBATCH --error=lhs_msel_{model}_{project_name}.err"+"\n")
            f.write("#SBATCH --time="+time+"\n")
            f.write("#SBATCH --ntasks=1\n") # 1 tasks per job
            f.write("#SBATCH --cpus-per-task=16\n")
            f.write("#SBATCH --chdir=/home/uc27/uc27001/glassboxevm/\n")
            f.write('export OPENBLAS_NUM_THREADS=1\n')
            f.write('module purge\n')
            f.write('source $HOME/mlenv/bin/activate\n')
            f.write(f"srun -n 1 --ntasks-per-node=16 python3 ./model_selection/model_selection.py {model} {project_name}")  
        f.close()


## FastAI selection re-run (SLURM arrays)

FastAI selection runs as one SLURM array per selection project, with one single-core task per LHS configuration (`model_selection.py fastai <project> --lhs <k>`). Constant inputs are dropped before training.

**Run in Altamira** (one subset per wave):

```bash
cd /gpfs/users/res/uc27/uc27001/glassboxevm
bash model_selection/submit_fastai_wave.sh J30   # then J60, J90, J120 when the previous wave has drained
python3 model_selection/merge_fastai_results.py  # merge + list of missing (project, LHS) + retry array
```

In [ ]:
# FastAI: one SLURM array per selection project, one short task per LHS
# configuration (task id = LHS index), 1 CPU, so that SLURM can schedule the tasks quickly.
# model_selection.py --lhs writes one CSV per (project, LHS) in results_fastai/; merge_fastai_results.py
# joins them into results/lhs_model_selection_fastai_project_<project>.csv.
# Submit by waves (one subset = 10 arrays = 500 tasks) with submit_fastai_wave.sh <SUBSET>,
# to stay below the QoS limit of 2000 submitted jobs per user.
from pathlib import Path

cluster_root = "/gpfs/users/res/uc27/uc27001/glassboxevm"
n_lhs = len(pd.read_csv("hyperparameters/lhs_fastai_parameters.csv"))
max_concurrent_tasks = 25
time_limit = "00:15:00"

jobs_dir = Path("./jobs")
jobs_dir.mkdir(exist_ok=True)
for file_name in df_files["FileName"].values:
    project_name = file_name.split(".")[0]
    script_path = jobs_dir / f"slurm_lhs_model_selection_fastai_array_{project_name}.sh"
    script_path.write_text(
        "#!/bin/bash\n"
        f"#SBATCH --job-name=msel_fastai_{project_name}\n"
        f"#SBATCH --output={cluster_root}/model_selection/slurm_logs/msel_fastai_{project_name}_%A_%a.out\n"
        f"#SBATCH --error={cluster_root}/model_selection/slurm_logs/msel_fastai_{project_name}_%A_%a.err\n"
        f"#SBATCH --time={time_limit}\n"
        "#SBATCH --mem=4G\n"
        f"#SBATCH --array=0-{n_lhs - 1}%{max_concurrent_tasks}\n"
        "#SBATCH --nodes=1\n"
        "#SBATCH --ntasks=1\n"
        "#SBATCH --cpus-per-task=1\n"
        "#SBATCH --hint=nomultithread\n"
        f"#SBATCH --chdir={cluster_root}\n"
        "\n"
        "export HOME=${HOME:-/gpfs/users/res/uc27/uc27001}\n"
        "export MPLBACKEND=Agg\n"
        "export CACHE_ROOT=/gpfs/users/res/uc27/uc27001/.cache\n"
        "export XDG_CACHE_HOME=${CACHE_ROOT}\n"
        "export FASTAI_HOME=${CACHE_ROOT}/fastai\n"
        "export TORCH_HOME=${CACHE_ROOT}/torch\n"
        "export MPLCONFIGDIR=${CACHE_ROOT}/matplotlib\n"
        "mkdir -p ${FASTAI_HOME} ${TORCH_HOME} ${MPLCONFIGDIR}\n"
        "export OPENBLAS_NUM_THREADS=${SLURM_CPUS_PER_TASK}\n"
        "export OMP_NUM_THREADS=${SLURM_CPUS_PER_TASK}\n"
        "export MKL_NUM_THREADS=${SLURM_CPUS_PER_TASK}\n"
        "export NUMEXPR_NUM_THREADS=${SLURM_CPUS_PER_TASK}\n"
        "\n"
        "type module &>/dev/null && module purge\n"
        "source /gpfs/users/res/uc27/uc27001/mlenv/bin/activate\n"
        "\n"
        "srun --ntasks=1 --cpus-per-task=${SLURM_CPUS_PER_TASK} --cpu-bind=cores "
        f"python3 ./model_selection/model_selection.py fastai {project_name} "
        "--lhs ${SLURM_ARRAY_TASK_ID} --path-results ./model_selection/results_fastai\n",
        encoding="utf-8",
        newline="\n",
    )

sorted(jobs_dir.glob("slurm_lhs_model_selection_fastai_array_*.sh"))[:3]
